##### 0. Setup the environment

In [1]:
from odp.client import Client

In [ ]:
api_key = "sk_your_api_key_here"
#client = Client(api_key = api_key)

# Or fall back to the ODP_API_KEY env var
client = Client()

##### 1. Read data

In [7]:
ds = client.dataset("21b630bb-06b2-48de-a172-97a7a67e30ba") # amazon reef
table = ds.table

In [ ]:
# Try to select all data as an Arrow Table
# NOTE: if the table is large, you might want to iterate over the select cursor instead
arrow = table.select().all().table()
gdf = table.select().all().dataframe()  # or as a Geopandas gdf

print(table.stats())
print(table.schema())

In [ ]:
# Select by column 'type'
gdf_coral = table.select(filter = "type == 'coral'").all().dataframe()
gdf_seagrass = table.select(filter = "type == 'seagrass'").all().dataframe()
gdf_rock = table.select(filter = "type == 'rock'").all().dataframe()

##### 2. Write data

In [ ]:
my_ds = client.dataset("your-dataset-uuid-here")
my_table = my_ds.table

In [ ]:
# Create and insert coral data frame into the new table
my_table.create(gdf_coral)
print(my_table.stats())
print(my_table.schema())

In [ ]:
# Insert more data (append)
with my_table as tx:
    tx.insert(gdf_seagrass)

In [ ]:
# For multi-step workflows, open a transaction
with my_table as tx:
    tx.delete(query = "type == 'coral'")
    tx.replace(query = "type == 'seagrass'", data = gdf_coral)
    tx.insert(gdf_rock)

#### 3. For more advanced features (like aggregation or schema manipulation), refer to: https://docs.hubocean.earth/